In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import sqlite3
import mysql.connector
import pyarrow

import numpy as np

In [6]:
connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")

MySQL DB Connected


In [7]:
cursor = connection.cursor()

cursor.execute("SELECT * FROM FT_DAILY_DATA WHERE Exchange = 'BTC-USD'")

results = cursor.fetchall()

columns = [column[0] for column in cursor.description]


df_original = pd.DataFrame(results, columns=columns)

In [8]:
df_original.sort_values(by = 'id_date', ascending = True)

,Open,High,Low,Close,adj_close,Volume,Exchange,id_exchange,id_date
0,466,468,452,457,457,21056800,BTC-USD,193,20140917
1,457,457,413,424,424,34483200,BTC-USD,193,20140918
2,424,428,385,395,395,37919700,BTC-USD,193,20140919
3,395,423,390,409,409,36863600,BTC-USD,193,20140920
4,408,412,393,399,399,26580100,BTC-USD,193,20140921
...,...,...,...,...,...,...,...,...,...
3527,62901,63092,61124,61553,61553,28186271527,BTC-USD,193,20240514
3528,61554,66454,61330,66267,66267,39815167074,BTC-USD,193,20240515
3529,66256,66712,64613,65232,65232,31573077994,BTC-USD,193,20240516
3530,65231,67459,65119,67052,67052,28031279310,BTC-USD,193,20240517


In [18]:
df = df_original

In [19]:
df['id_date'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
df = df[['Open', 'High', 'Low', 'Close', 'adj_close', 'Volume', 'id_date']]
df

,Open,High,Low,Close,adj_close,Volume,id_date
0,466,468,452,457,457,21056800,2014-09-17
1,457,457,413,424,424,34483200,2014-09-18
2,424,428,385,395,395,37919700,2014-09-19
3,395,423,390,409,409,36863600,2014-09-20
4,408,412,393,399,399,26580100,2014-09-21
...,...,...,...,...,...,...,...
3527,62901,63092,61124,61553,61553,28186271527,2024-05-14
3528,61554,66454,61330,66267,66267,39815167074,2024-05-15
3529,66256,66712,64613,65232,65232,31573077994,2024-05-16
3530,65231,67459,65119,67052,67052,28031279310,2024-05-17


In [20]:
from typing import Tuple

def train_test_split(
    df: pd.DataFrame,
    cutoff_date: datetime,
    target_column_name: str,
    ) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    """
    train_data = df[df.id_date < cutoff_date].reset_index(drop=True)
    test_data = df[df.id_date >= cutoff_date].reset_index(drop=True)

    X_train = train_data.drop(columns=[target_column_name])
    y_train = train_data[target_column_name]
    X_test = test_data.drop(columns=[target_column_name])
    y_test = test_data[target_column_name]

    return X_train, y_train, X_test, y_test

In [35]:
from datetime import datetime

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date=datetime(2024, 4, 1, 0, 0, 0),
    target_column_name='Close'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')

X_train.shape=(3484, 6)
y_train.shape=(3484,)
X_test.shape=(48, 6)
y_test.shape=(48,)


In [38]:
X_train = X_train.astype(int)
X_test = X_test.astype(int)
y_train = y_train.astype(int)
y_test = y_test.astype(int)


X_train_only_numeric = X_train.drop(columns = 'id_date')


X_train_only_numeric.head(5)

,Open,High,Low,adj_close,Volume
0,466,468,452,457,21056800
1,457,457,413,424,34483200
2,424,428,385,395,37919700
3,395,423,390,409,36863600
4,408,412,393,399,26580100


In [37]:
import xgboost as xgb

In [24]:
model = xgb.XGBRegressor()

In [33]:
model.fit(X_train_only_numeric, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [39]:
X_test_only_numeric = X_test.drop(columns = 'id_date')
predictions = model.predict(X_test_only_numeric)
predictions

array([69083.77 , 66251.18 , 67069.42 , 68777.49 , 66772.29 , 69241.805,
       70222.47 , 69394.77 , 69468.57 , 69375.9  , 70530.08 , 64876.676,
       63911.793, 66248.2  , 63546.92 , 64307.24 , 61692.844, 63748.75 ,
       64304.336, 64096.293, 63424.79 , 66052.16 , 65843.36 , 63623.23 ,
       64460.004, 64083.965, 63461.145, 62265.582, 64337.12 , 61385.12 ,
       58449.254, 59249.336, 62296.996, 63922.188, 63936.01 , 62672.85 ,
       62293.52 , 61203.633, 61960.367, 61577.184, 61377.06 , 61333.03 ,
       62579.695, 61627.773, 66214.336, 66998.625, 66052.16 , 66250.6  ],
      dtype=float32)

In [40]:
from sklearn.metrics import mean_absolute_error
test_mae = mean_absolute_error(y_test, predictions)
print(f'{test_mae=:.4f}')

test_mae=602.0232


#### LIGHTGBM

In [42]:
import lightgbm as lgb

In [43]:
model_lgb = lgb.LGBMRegressor()
model_lgb.fit(X_train_only_numeric, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000243 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 3484, number of used features: 5
[LightGBM] [Info] Start training from score 15601.627727


LGBMRegressor()

In [45]:
X_test_only_numeric = X_test.drop(columns = 'id_date')
predictions = model_lgb.predict(X_test_only_numeric)

from sklearn.metrics import mean_absolute_error
test_mae = mean_absolute_error(y_test, predictions)
print(f'{test_mae=:.4f}')

test_mae=610.5154
